##### NanoMaker use / execution workflow
This notebook will allow you to generate protein binding pockets for any chemical provided in SMILES format. The comments in each of the cells will walk you through.

In [1]:
from src.nano_maker.utility import *
from src.nano_maker.container.configs import *
from src.nano_maker.nanomaker import NanoMaker

In [2]:
# initialize NanoMaker system
model = NanoMaker(
    version_code=version_code,
    skeleton_weight_filename=skeleton_weight_filename,
    naanobot_weight_filename=naanobot_weight_filename,
    skeleton_cfg=skeleton_config,
    naanobot_cfg=naanobot_config,
    radial_cfg=radial_config)

In [10]:
# pt 1 -> let's create a binding pocket for a given drug
# - generate the binding pocket data
# - then save it to a .nanopkt file format
drug_i_want_to_deliver = "OC[C@@H](O)[C@H]1OC(=O)C(O)=C1O"   # Vitamin C
model.ingest_chemical(drug_i_want_to_deliver)

# you won't always need to save it to a file format but it's handy.

Chemical Ingested: OC[C@@H](O)[C@H]1OC(=O)C(O)=C1O
Scaffold: O=C1C=CCO1


In [11]:
pocket_data = model.generate_nanopkt_data(sampling_temp=0.3)
# sampling temperature is on a 0-1 float scale -> determines amino acid sampling strictness
# lower <= 0.2 -> fully optimized pockets but not realistic
# balanced 0.2 < 'x' <= 0.5  ->  balance between biochemical optimization and diversity
# higher 0.5 < 'x' <= 1 -> flatter prob. distribution, exploration-oriented
# essentially random: 1+
print(type(pocket_data))

<class 'dict'>


In [17]:
filename= "vitamin_C_pocket"
save_nano_pocket(pocket_data=pocket_data, filename=filename)

Pocket data for OC[C@@H](O)[C@H]1OC(=O)C(O)=C1O saved:
Find nano pocket file in: <_io.TextIOWrapper name='/home/elliot/Desktop/LOCAL_PROJECTS/nano_maker/src/output_container/vitamin_C_pocket.nanopkt' mode='w' encoding='UTF-8'>


True

In [18]:
nanopkt_data = load_nano_pocket(filename=filename)
if nanopkt_data == pocket_data:
    print("Data Aligned")
# ensure the file saving / loading workflow is consistent in loading the data

Data Aligned


Now that we have generated a "nanopocket" for a given smiles and its data is saved, we can visualize the generated protein binding pocket with the following cell / module.

In [22]:
from nano_maker.pocketwatcher import PocketWatcher

visualizer = PocketWatcher(naanoeng_config=naanoeng_config, pocket_config=pocketwatcher_config)
visualizer.ingest_file(pocket_filepath=filename)
visualizer.visualize_pocket(identifier="steric_accessibility")
# the identifier parameter allows users to distinguish the pocket's amino acids from each other either by plain cosmetic or by their biochemical properties. These are actually "summaries" of many features from the feature vectors engineered for NAAnoBot.

# cosmetic identifier settings
# [1] 'skeleton' -> just shows amino acids as white
# [2] 'color_code' -> color codes amino acids

# biochemical property identifier settings
# [3] 'polar_character' -> summary metric of AA's electrical environment, greater = polar
# [4] 'hydrophobicity' -> greater = hydrophobic, lower = hydrophilic
# [5] 'flexibility' -> composite metric of AA's flexibility, greater = more freedom, lower = restricted
# [6] 'steric_accessibility' -> AA's size, weight, and exposed surface, greater = larger / more exposed

PocketWatcher also allows one to generate reports on the generated pockets for detailed quantitative analyses on the produced pockets.

In [20]:
print(visualizer.pocket_report())

BINDING POCKET REPORT
Sampling temperature: 0.3
Amino acid sequence:ESKGFCRAIAYVYWDNVKKAGAFPCNTSGGDFGGEYH

Section 1: Biochemical Property Summary
|-- average_polar_character: 0.180
|-- polar_character_deviation_pct: -2.439
|-- average_hydrophobicity: 0.179
|-- hydrophobicity_deviation_pct: -22.143
|-- average_flexibility: 3.811
|-- flexibility_deviation_pct: 17.529
|-- average_steric: 2.176
|-- steric_deviation_pct: -5.504

Section 2: Geometric Analysis
|-- X range: 16.490
|-- Y range: 11.671
|-- Z range: 6.252


Notable Binding Pocket Characteristics:
- moderately hydrophilic, slightly flexible, surface patch-style binding pocket


I should mention that the "notable characteristics" section at the bottom of each report are all computed relative to the average properties across all 20 amino acids. So when we read "moderately hydrophilic", its saying that the average hydrophobicity of all amino acids in that binding pocket is slightly lower than the average hydrophobicity across all 20 amino acids.